In [2]:
import nibabel as nib

# Path to your BOLD file
bold_path = "/juice6/u/nlp/climblab/BIDS/climblab_multisession/sub-c001/ses-ex32061/func/sub-c001_ses-ex32061_task-listeningLoc_run-01_bold.nii.gz"

# Load the NIfTI image
img = nib.load(bold_path)

# Get voxel size (in mm) for x, y, z
voxel_size = img.header.get_zooms()[:3]

print("Voxel size (x, y, z) in mm:", voxel_size)

Voxel size (x, y, z) in mm: (np.float32(2.0175), np.float32(2.0175), np.float32(2.0))


In [15]:
import pandas as pd
bold_filename = '/juice6/u/nlp/climblab/BIDS/climblab_multisession/derivatives/preprocess/main/sub-c001/ses-ex31886/func/sub-c001_ses-ex31886_task-wordseqcovert_run-02_space-T1w_desc-preproc_bold.nii.gz'
confounds_path = bold_filename.replace("_space-T1w_desc-preproc_bold.nii.gz", "_desc-confounds_timeseries.tsv")
#confounds_regex = r'^(?:trans|rot)_[xyz](?:$|_(?:derivative1|power2|derivative1_power2)$)|global_signal(?:$|_derivative1|_power2|_derivative1_power2)$|a_comp_cor_.*|non_steady_state_outlier.*|motion_outlier.*|framewise_displacement$'
#confounds_regex = r'^(?:trans|rot)_[xyz](?:$|_(?:derivative1|power2)$)|global_signal(?:$|_derivative1|_power2|_derivative1_power2)$'
confounds_regex = r'^(?:trans|rot)_[xyz](?:$|_(?:derivative1|power2)$)|global_signal(?:$|_derivative1|_power2)$|a_comp_cor_0[0-4]$|non_steady_state_outlier.*|motion_outlier.*|framewise_displacement$'

confounds = pd.read_csv(confounds_path, sep='\t')
confounds = confounds.filter(regex=confounds_regex)
confounds = confounds.fillna(0)

In [16]:
confounds

,global_signal,global_signal_derivative1,global_signal_power2,framewise_displacement,a_comp_cor_00,a_comp_cor_01,a_comp_cor_02,a_comp_cor_03,a_comp_cor_04,non_steady_state_outlier00,...,rot_y_power2,rot_z,rot_z_derivative1,rot_z_power2,motion_outlier00,motion_outlier01,motion_outlier02,motion_outlier03,motion_outlier04,motion_outlier05
0,2411.719873,0.000000,5.816393e+06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,...,6.282813e-09,-0.000155,0.000000,2.403554e-08,0.0,0.0,0.0,0.0,0.0,0.0
1,1929.621315,-482.098558,3.723438e+06,0.553482,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,4.875391e-07,0.000000,0.000155,0.000000e+00,1.0,0.0,0.0,0.0,0.0,0.0
2,1751.714320,-177.906995,3.068503e+06,0.151905,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,7.978277e-07,-0.000459,-0.000459,2.103406e-07,0.0,1.0,0.0,0.0,0.0,0.0
3,1679.142911,-72.571409,2.819521e+06,0.063490,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,1.043585e-06,-0.000277,0.000182,7.673953e-08,0.0,0.0,0.0,0.0,0.0,0.0
4,1646.596393,-32.546518,2.711280e+06,0.154937,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,1.520610e-06,0.000143,0.000420,2.038870e-08,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
368,1609.076601,-1.164845,2.589128e+06,0.237388,-0.019327,-0.011680,-0.066884,0.185909,-0.023401,0.0,...,4.295008e-06,-0.000122,0.000576,1.487278e-08,0.0,0.0,0.0,0.0,0.0,0.0
369,1610.930532,1.853931,2.595097e+06,0.200230,-0.003513,-0.000253,-0.051706,0.167179,-0.048397,0.0,...,8.564227e-06,0.000215,0.000337,4.635194e-08,0.0,0.0,0.0,0.0,0.0,0.0
370,1614.050665,3.120133,2.605160e+06,0.132732,0.012276,0.006125,-0.059652,0.138470,0.001295,0.0,...,8.701674e-06,0.000068,-0.000148,4.576753e-09,0.0,0.0,0.0,0.0,0.0,0.0
371,1614.182425,0.131759,2.605585e+06,0.045118,0.039805,-0.044470,-0.085041,0.139527,-0.009229,0.0,...,9.882913e-06,0.000179,0.000112,3.213594e-08,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#--load ICs--
manual_signals = np.loadtxt(os.path.join(ica_root, 'signals.txt')).astype(np.int32)
ic_path = os.path.join(ica_root, 'melodic', 'melodic_IC.nii.gz')
ic_img = nib.load(str(ic_path))
ics    = ic_img.get_fdata(dtype=np.float32)         # shape: (X,Y,Z,K)

K = ics.shape[3]
Amat = ics.reshape(-1, K).copy()        # (V, K)
Amat = Amat[:,manual_signals-1]
A_pinv = np.linalg.pinv(Amat, rcond=1e-6)

A_all   = ics.reshape(-1, K).copy()                    # (V, K) all ICs
Aall_pinv = np.linalg.pinv(A_all, rcond=1e-6)          # (K, V)
noise_idx = np.setdiff1d(np.arange(0,K).astype(np.int32), manual_signals-1)

#--project off noise ICs--
bold_img = nib.load(str(hp_file) + '.nii.gz')
bold     = bold_img.get_fdata(dtype=np.float32)     # shape: (X,Y,Z,T)
aff      = bold_img.affine
hdr      = bold_img.header.copy()

X, Y, Z, T = bold.shape        
K = ics.shape[3]
Ymat = bold.reshape(-1, T)       # (V, T)
V = Amat.shape[0]

#normalize bold
Ymat[np.isnan(Ymat)]=0
Ymat = scipy.stats.zscore(Ymat, axis=1)
Ymat[np.isnan(Ymat)]=0

#subtract noise
C_all = Aall_pinv @ Ymat
Y_noise = A_all[:, noise_idx] @ C_all[noise_idx, :]
Ymat = Ymat - Y_noise